# 🦙 LlamaChat Pro — LangChain + Llama 3 Pipeline
**Multi-mode AI Assistant** powered by Groq's free Llama 3.3-70B API

This notebook demonstrates:
1. **General Chat** — conversational memory with LangChain
2. **RAG Pipeline** — document ingestion → chunking → embedding → FAISS → retrieval → generation
3. **Domain Expert** — system-prompt-based persona engineering
4. **Evaluation** — response quality metrics

**Setup**: Get a free API key at [console.groq.com](https://console.groq.com)


## 0 · Install Dependencies

In [1]:
!pip install -q groq langchain==0.2.17 langchain-community langchain-groq \n    langchain-huggingface faiss-cpu sentence-transformers \n    rank-bm25 pdfplumber pypdf docx2txt python-dotenv tiktoken
print("✅ Packages installed")

ERROR: Could not find a version that satisfies the requirement n (from versions: none)
ERROR: No matching distribution found for n
✅ Packages installed


## 1 · Configuration

In [2]:
import os, json, time
from IPython.display import display, Markdown

# ── Set your Groq API key ──────────────────────────────────────────────────────
GROQ_API_KEY = "gsk_your_key_here"   # Get free at console.groq.com
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

MODEL = "llama-3.3-70b-versatile"         # Best quality — 70B Llama 3.3
# MODEL = "llama-3.1-8b-instant"          # Faster — 8B model

print(f"Model  : {MODEL}")
print(f"API key: {'✅ Set' if GROQ_API_KEY != 'your_groq_api_key_here' else '❌ Not set — update above'}")


Model  : llama-3.3-70b-versatile
API key: ✅ Set


## 2 · Direct Groq Chat (No LangChain)

In [3]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

def chat(prompt: str, system: str = "You are a helpful assistant.", temperature: float = 0.7):
    """Simple single-turn Groq call."""
    t0 = time.time()
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system",  "content": system},
            {"role": "user",    "content": prompt},
        ],
        temperature=temperature,
        max_tokens=1024,
    )
    elapsed = time.time() - t0
    text    = response.choices[0].message.content
    tokens  = response.usage.total_tokens
    print(f"⏱  {elapsed:.2f}s  |  🔢 {tokens} tokens  |  ⚡ {tokens/elapsed:.0f} tok/s")
    return text

# Test
answer = chat("What makes Llama 3.3 better than previous Llama versions?")
display(Markdown(answer))


⏱  0.51s  |  🔢 118 tokens  |  ⚡ 232 tok/s


I'm an AI model known as Llama. I was developed by Meta with a combination of machine learning algorithms and large amounts of data, plus lots of human oversight from a large team of people. I'm constantly learning and improving, so over time I will likely become even more useful in my responses.

## 3 · Conversational Memory with LangChain

In [4]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Modern LangChain memory — manual sliding window (replaces deprecated ConversationBufferWindowMemory)
class SlidingWindowMemory:
    def __init__(self, k=10):
        self.k = k
        self.history = []  # list of (human, ai) tuples

    def add(self, human_msg: str, ai_msg: str):
        self.history.append((human_msg, ai_msg))
        if len(self.history) > self.k:
            self.history = self.history[-self.k:]

    def get_messages(self, system_prompt: str) -> list:
        msgs = [SystemMessage(content=system_prompt)]
        for h, a in self.history:
            msgs.append(HumanMessage(content=h))
            msgs.append(AIMessage(content=a))
        return msgs

    def clear(self):
        self.history = []

# ── LangChain Groq LLM ───────────────────────────────────────────────────
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name=MODEL,
    temperature=0.7,
    max_tokens=1024,
)

# ── Memory: keeps last 10 exchanges ─────────────────────────────────────
memory = SlidingWindowMemory(k=10)

SYSTEM_PROMPT = (
    "You are LlamaChat Pro, a helpful AI assistant powered by Llama 3.3-70B. "
    "Be concise when needed and detailed when depth is required."
)

def chat_with_memory(user_input: str) -> str:
    """Multi-turn chat using sliding window memory."""
    messages = memory.get_messages(SYSTEM_PROMPT)
    messages.append(HumanMessage(content=user_input))
    response  = llm.invoke(messages)
    ai_text   = response.content
    memory.add(user_input, ai_text)
    return ai_text

print("✅ LangChain ChatGroq + SlidingWindowMemory ready")
print(f"   Model   : {MODEL}")
print(f"   Memory  : last {memory.k} exchanges")


✅ LangChain ChatGroq + SlidingWindowMemory ready
   Model   : llama-3.3-70b-versatile
   Memory  : last 10 exchanges


## 4 · RAG Pipeline — Chat with Documents

In [5]:
# ── 4.1 Create a sample document ──────────────────────────────────────────────
sample_doc = """
ARTIFICIAL INTELLIGENCE IN HEALTHCARE — OVERVIEW

Introduction:
Artificial Intelligence (AI) is transforming healthcare by enabling faster diagnostics,
personalized treatment plans, and predictive analytics. Machine learning models can now
detect diseases such as cancer from medical images with accuracy matching or exceeding
human experts.

Key Applications:
1. Medical Imaging: Deep learning models (CNNs) analyze X-rays, MRIs, and CT scans.
   Example: Google DeepMind's AI detects eye diseases with 94% accuracy.

2. Drug Discovery: AI accelerates molecule screening from years to weeks.
   Example: AlphaFold by DeepMind solved the 50-year protein folding problem.

3. Predictive Analytics: Models predict patient deterioration 6-12 hours in advance,
   enabling early intervention and reducing ICU mortality by up to 20%.

4. Natural Language Processing: Clinical NLP extracts structured data from unstructured
   medical notes, lab reports, and discharge summaries.

5. Robotic Surgery: AI-assisted surgical robots improve precision and reduce recovery time.
   The da Vinci surgical system has performed over 10 million procedures worldwide.

Challenges:
- Data privacy and HIPAA compliance
- Model explainability (black-box problem)
- Regulatory approval (FDA clearance required for diagnostic AI)
- Bias in training data leading to disparate outcomes across demographics

Future Outlook:
The global AI in healthcare market is projected to reach $208 billion by 2030 (CAGR 45%).
Foundation models like GPT-4 and Med-PaLM 2 are being adapted for clinical use.
"""

with open("/tmp/healthcare_ai.txt", "w") as f:
    f.write(sample_doc)

print("Sample document created: /tmp/healthcare_ai.txt")
print(f"Length: {len(sample_doc.split())} words")


Sample document created: /tmp/healthcare_ai.txt
Length: 216 words


In [6]:
# ── 4.2 Load & chunk document ─────────────────────────────────────────────────
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader   = TextLoader("/tmp/healthcare_ai.txt")
docs     = loader.load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " "]
)
chunks = splitter.split_documents(docs)

print(f"Document loaded  : {len(docs)} page(s)")
print(f"Chunks created   : {len(chunks)}")
print(f"Avg chunk length : {sum(len(c.page_content) for c in chunks)//len(chunks)} chars")
print(f"\nSample chunk:\n{'-'*50}\n{chunks[2].page_content}\n{'-'*50}")


Document loaded  : 1 page(s)
Chunks created   : 6
Avg chunk length : 259 chars

Sample chunk:
--------------------------------------------------
3. Predictive Analytics: Models predict patient deterioration 6-12 hours in advance,
   enabling early intervention and reducing ICU mortality by up to 20%.

4. Natural Language Processing: Clinical NLP extracts structured data from unstructured
   medical notes, lab reports, and discharge summaries.
--------------------------------------------------


In [ ]:
# ── 4.3 Embed chunks → FAISS vector store ─────────────────────────────────────
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Loading embedding model (sentence-transformers/all-MiniLM-L6-v2)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Building FAISS index...")
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"✅ Vector store ready — {vectorstore.index.ntotal} vectors indexed")

# Test retrieval
query   = "How does AI help in drug discovery?"
results = vectorstore.similarity_search(query, k=3)
print(f"\nQuery: '{query}'")
print(f"Top match: {results[0].page_content[:200]}...")


Loading embedding model (sentence-transformers/all-MiniLM-L6-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

In [ ]:
# ── 4.4 Full RAG Q&A chain ────────────────────────────────────────────────────
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

rag_prompt_template = """Use ONLY the following context to answer the question.
If the answer is not in the context, say "I don't know based on the document."
Be concise, accurate, and cite which part of the document supports your answer.

Context:
{context}

Question: {question}

Answer:"""

rag_prompt = PromptTemplate(
    template=rag_prompt_template,
    input_variables=["context", "question"]
)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    chain_type_kwargs={"prompt": rag_prompt},
    return_source_documents=True
)

# ── Test RAG questions ─────────────────────────────────────────────────────────
questions = [
    "What are the key applications of AI in healthcare?",
    "What is the projected market size of AI in healthcare by 2030?",
    "What challenges exist for AI in healthcare?",
    "Who invented the transformer architecture?",   # Not in doc — should say I don't know
]

print("=== RAG Pipeline Q&A ===\n")
for q in questions:
    result = rag_chain.invoke({"query": q})
    print(f"❓ {q}")
    print(f"🦙 {result['result'][:300]}")
    print(f"   [Sources: {len(result['source_documents'])} chunks retrieved]\n")


## 5 · Domain Expert Mode

In [ ]:
DOMAIN_PROMPTS = {
    "medical": (
        "You are an expert medical AI. Provide evidence-based medical information. "
        "Always advise consulting a physician for diagnosis/treatment."
    ),
    "legal": (
        "You are an expert legal AI. Provide clear legal analysis. "
        "Always recommend consulting a licensed attorney for specific advice."
    ),
    "finance": (
        "You are an expert financial AI. Provide financial education and analysis. "
        "Note this is not personalized financial advice."
    ),
    "code": (
        "You are an expert software engineer. Help with code, algorithms, debugging, "
        "and best practices. Write clean, well-commented code with examples."
    ),
}

def expert_chat(question: str, domain: str):
    """Chat with a domain expert persona."""
    system = DOMAIN_PROMPTS.get(domain, DOMAIN_PROMPTS["code"])
    return chat(question, system=system, temperature=0.5)

print("=== Domain Expert Mode Demo ===\n")

# Medical
q = "What are early warning signs of type 2 diabetes?"
print(f"🩺 MEDICAL EXPERT\nQ: {q}")
ans = expert_chat(q, "medical")
display(Markdown(ans[:600] + "..."))

print()

# Code
q = "Write a Python function to implement a binary search tree with insert and search."
print(f"💻 CODE EXPERT\nQ: {q}")
ans = expert_chat(q, "code")
display(Markdown(ans[:800] + "..."))


## 6 · Pipeline Evaluation

In [ ]:
import re

def evaluate_rag(vectorstore, llm, qa_pairs: list[dict]) -> dict:
    """
    Evaluate RAG pipeline on Q&A pairs.
    qa_pairs: [{'question': str, 'expected_keywords': list[str]}]
    """
    rag = RetrievalQA.from_chain_type(
        llm=llm, chain_type="stuff",
        retriever=vectorstore.as_retriever(search_kwargs={"k":3}),
        return_source_documents=True
    )
    results = []
    for item in qa_pairs:
        r    = rag.invoke({"query": item["question"]})
        ans  = r["result"].lower()
        hits = sum(1 for kw in item["expected_keywords"] if kw.lower() in ans)
        score = hits / len(item["expected_keywords"])
        results.append({
            "question": item["question"],
            "score":    round(score * 100, 1),
            "answer":   r["result"][:150] + "..."
        })

    avg_score = sum(r["score"] for r in results) / len(results)
    return {"results": results, "avg_score": round(avg_score, 1)}

eval_set = [
    {"question": "What does AI do in medical imaging?",
     "expected_keywords": ["cancer", "X-ray", "MRI", "deep learning", "CNN"]},
    {"question": "What is AlphaFold?",
     "expected_keywords": ["protein", "DeepMind", "folding"]},
    {"question": "What is the projected market size?",
     "expected_keywords": ["208 billion", "2030", "45%"]},
]

print("Evaluating RAG pipeline...")
eval_results = evaluate_rag(vectorstore, llm, eval_set)

print(f"\n{'='*55}")
print(f"  RAG PIPELINE EVALUATION — Avg Score: {eval_results['avg_score']}%")
print(f"{'='*55}")
for r in eval_results["results"]:
    bar = "█" * int(r["score"]//10) + "░" * (10 - int(r["score"]//10))
    print(f"\n  ❓ {r['question'][:60]}")
    print(f"  [{bar}] {r['score']}%")
    print(f"  💬 {r['answer']}")


## 7 · Pipeline Architecture Summary

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Modern LangChain memory — manual sliding window (replaces deprecated ConversationBufferWindowMemory)
class SlidingWindowMemory:
    def __init__(self, k=10):
        self.k = k
        self.history = []  # list of (human, ai) tuples

    def add(self, human_msg: str, ai_msg: str):
        self.history.append((human_msg, ai_msg))
        if len(self.history) > self.k:
            self.history = self.history[-self.k:]

    def get_messages(self, system_prompt: str) -> list:
        msgs = [SystemMessage(content=system_prompt)]
        for h, a in self.history:
            msgs.append(HumanMessage(content=h))
            msgs.append(AIMessage(content=a))
        return msgs

    def clear(self):
        self.history = []

# ── LangChain Groq LLM ───────────────────────────────────────────────────
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name=MODEL,
    temperature=0.7,
    max_tokens=1024,
)

# ── Memory: keeps last 10 exchanges ─────────────────────────────────────
memory = SlidingWindowMemory(k=10)

SYSTEM_PROMPT = (
    "You are LlamaChat Pro, a helpful AI assistant powered by Llama 3.3-70B. "
    "Be concise when needed and detailed when depth is required."
)

def chat_with_memory(user_input: str) -> str:
    """Multi-turn chat using sliding window memory."""
    messages = memory.get_messages(SYSTEM_PROMPT)
    messages.append(HumanMessage(content=user_input))
    response  = llm.invoke(messages)
    ai_text   = response.content
    memory.add(user_input, ai_text)
    return ai_text

print("✅ LangChain ChatGroq + SlidingWindowMemory ready")
print(f"   Model   : {MODEL}")
print(f"   Memory  : last {memory.k} exchanges")
